# Modi OCR كامل — YOLO11s متعدد الكلاسات


1. تثبيت المكتبات وربط Google Drive
2. إعداد المسارات
3. تضمين وفحص الخطوط
4. تضمين القاموس الخام واستخراج graphemes
5. توليد الصور والوسوم الخام
6. تحويل البيانات وتقسيمها
7. نسخ البيانات إلى كاش محلي
8. تدريب YOLO11s
9. التقييم والتنبؤ
10. واجهة Gradio لإخراج النص

كل grapheme مدعوم له **كلاس مستقل** حتى يستطيع YOLO تحديد الحرف ثم تكوين النص.


In [ ]:
%pip install -q --upgrade-strategy only-if-needed "ultralytics>=8.3.0,<9" "regex>=2024.0" "pyyaml>=6,<7" "scikit-learn>=1.4,<2" "tqdm>=4.66" "onnx>=1.16,<1.20" "onnxruntime>=1.18,<1.21"

from google.colab import drive
from pathlib import Path
import os, json, random, shutil, subprocess, math
import numpy as np
import cv2
from PIL import Image, ImageDraw, ImageFont
from tqdm.auto import tqdm

drive.mount('/content/drive')
print('تم ربط Google Drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
تم ربط Google Drive


In [ ]:
# الخلية 2 — إعداد المسارات
PROJECT = Path('/content/drive/MyDrive/modi_ocr_yolo11s_project')
RAW_IMAGES = PROJECT / 'raw_images'
RAW_LABELS = PROJECT / 'raw_labels'
DATASET = PROJECT / 'dataset'
RUNS = PROJECT / 'runs'
EXPORTS = PROJECT / 'exports'
APK_OUT = PROJECT / 'apk'
FONT_DIR = PROJECT / 'fonts'
DATA_DIR = PROJECT / 'data'

for p in [RAW_IMAGES, RAW_LABELS, DATASET/'images/train', DATASET/'images/val', DATASET/'images/test',
          DATASET/'labels/train', DATASET/'labels/val', DATASET/'labels/test', DATASET/'annotations',
          RUNS, EXPORTS, APK_OUT, FONT_DIR, DATA_DIR]:
    p.mkdir(parents=True, exist_ok=True)

SEED = 42
random.seed(SEED); np.random.seed(SEED)
rng = random.Random(SEED)
N_SAMPLES = 5000
OVERWRITE = False
IMAGE_SIZE = (1268, 463)
print('PROJECT =', PROJECT)
print('ضع الخطوط في:', FONT_DIR)
print('ضع القاموس الخام الاختياري في:', DATA_DIR/'modi.text')


PROJECT = /content/drive/MyDrive/modi_ocr_yolo11s_project
ضع الخطوط في: /content/drive/MyDrive/modi_ocr_yolo11s_project/fonts
ضع القاموس الخام الاختياري في: /content/drive/MyDrive/modi_ocr_yolo11s_project/data/modi.text


In [ ]:
# الخلية 3 — تضمين الخطوط وفحص دعم حروف Modi
FONT_FILES = sorted(list(FONT_DIR.glob('*.ttf')) + list(FONT_DIR.glob('*.otf')))
if not FONT_FILES:
    raise FileNotFoundError('ارفع خط Modi بصيغة TTF أو OTF إلى: ' + str(FONT_DIR))

MODI_BLOCK = '𑘀𑘁𑘂𑘃𑘄𑘅𑘆𑘇𑘈𑘉𑘊𑘋𑘌𑘍𑘎𑘏𑘐𑘑𑘒𑘓𑘔𑘕𑘖𑘗𑘘𑘙𑘚𑘛𑘜𑘝𑘞𑘟𑘠𑘡𑘢𑘣𑘤𑘥𑘦𑘧𑘨𑘩𑘪𑘫𑘬𑘭𑘮𑘯𑘰𑘱𑘲𑘳𑘴𑘵𑘶𑘷𑘸𑘹𑘺𑘻𑘼𑘽𑘾𑘿'
try:
    import regex
    def graphemes(s): return [g for g in regex.findall(r'\X', str(s)) if not g.isspace()]
except Exception:
    def graphemes(s): return [c for c in str(s) if not c.isspace()]

VALID = set()
for fp in FONT_FILES:
    font = ImageFont.truetype(str(fp), 96)
    for ch in graphemes(MODI_BLOCK):
        mask = Image.new('L', (240, 180), 0)
        ImageDraw.Draw(mask).text((8, 8), ch, font=font, fill=255)
        if mask.getbbox(): VALID.add(ch)
print('الخطوط:', [p.name for p in FONT_FILES])
print('الرموز القابلة للرسم:', len(VALID))


الخطوط: ['NotoSansModi-Bold.ttf', 'NotoSansModi-Regular.ttf']
الرموز القابلة للرسم: 48


In [ ]:
# الخلية 4 — تضمين القاموس الخام واستخراج الحروف/classes
MODI_TEXTS = [
    '𑘦𑘨𑘰𑘘𑘲 𑘦𑘳𑘩𑘰𑘦𑘰',
    '𑘭𑘰𑘮𑘱𑘡𑘳 𑘩𑘱𑘦𑘱',
    '𑘮𑘰𑘨𑘰𑘬𑘰 𑘦𑘾𑘨𑘰𑘚𑘱',
    '𑘩𑘱𑘦𑘱 𑘨𑘰𑘕𑘰 𑘦𑘨𑘰𑘘𑘱',
    '𑘦𑘨𑘰𑘘𑘱 𑘨𑘰𑘕𑘰 𑘭𑘰𑘮𑘱𑘡𑘳',
]
CORPUS_CANDIDATES = [DATA_DIR/'modi.text', DATA_DIR/'modi-training.text']
corpus_path = next((p for p in CORPUS_CANDIDATES if p.exists()), CORPUS_CANDIDATES[0])
def is_modi(s): return any(0x11600 <= ord(c) <= 0x116FF for c in str(s))
if corpus_path.exists():
    extra = [x.strip() for x in corpus_path.read_text(encoding='utf-8', errors='ignore').splitlines() if x.strip() and is_modi(x)]
    MODI_TEXTS += extra[:12000]
    print('تم استخدام القاموس:', corpus_path.name)
    print('تمت إضافة أسطر القاموس الخام:', len(extra[:12000]))
else:
    print('لم يوجد modi.text أو modi-training.text؛ سيُستخدم القاموس المضمّن فقط.')
MODI_TEXTS = list(dict.fromkeys(MODI_TEXTS))
MODI_CHARACTERS = sorted(set(g for line in MODI_TEXTS for g in graphemes(line)) & VALID, key=lambda x: ord(x[0]))
if len(MODI_CHARACTERS) < 5: raise RuntimeError('القاموس قليل أو الخط لا يدعم الرموز المطلوبة.')
CLASS_TO_ID = {ch:i for i,ch in enumerate(MODI_CHARACTERS)}
ID_TO_CLASS = {i:ch for ch,i in CLASS_TO_ID.items()}
(EXPORTS/'classes.json').write_text(json.dumps({str(i):ch for i,ch in ID_TO_CLASS.items()}, ensure_ascii=False, indent=2), encoding='utf-8')
print('عدد الأسطر:', len(MODI_TEXTS))
print('عدد الكلاسات:', len(MODI_CHARACTERS))
print('الحروف:', ''.join(MODI_CHARACTERS))


تم استخدام القاموس: modi-training.text
تمت إضافة أسطر القاموس الخام: 12000
عدد الأسطر: 12005
عدد الكلاسات: 40
الحروف: 𑘀𑘁𑘃𑘄𑘊𑘋𑘌𑘍𑘎𑘏𑘐𑘑𑘓𑘔𑘕𑘖𑘗𑘘𑘙𑘚𑘛𑘜𑘝𑘞𑘟𑘠𑘡𑘢𑘣𑘤𑘥𑘦𑘧𑘨𑘩𑘪𑘫𑘬𑘭𑘮


In [ ]:
# الخلية 5 — توليد بيانات Modi


N_SAMPLES = 5000
OVERWRITE = True


def paper_background(width, height):
    """
    خلفية ورق عشوائية وخفيفة وسريعة.
    """

    paper_palette = [
        (246, 243, 232),
        (239, 235, 218),
        (232, 229, 214),
        (248, 246, 238),
        (242, 234, 216),
        (235, 228, 207),
    ]

    base_color = np.array(
        rng.choice(paper_palette),
        dtype=np.float32
    )

    # ضوضاء خفيفة
    noise_rng = np.random.default_rng(
        rng.randrange(2**32)
    )

    noise = noise_rng.normal(
        0,
        rng.uniform(1.0, 3.0),
        (height, width, 1)
    )

    array = np.clip(
        base_color + noise,
        0,
        255
    ).astype(np.uint8)

    image = Image.fromarray(array)
    draw = ImageDraw.Draw(image, "RGBA")

    # خطوط دفتر عشوائية
    line_gap = rng.randint(72, 105)
    first_line = rng.randint(15, 35)

    for y in range(
        first_line,
        height,
        line_gap
    ):
        draw.line(
            [
                (0, y),
                (width, y + rng.randint(-1, 1))
            ],
            fill=rng.choice([
                (70, 70, 70, 70),
                (80, 90, 120, 65),
                (100, 80, 60, 55),
            ]),
            width=1
        )

    # بقع خفيفة
    for _ in range(rng.randint(0, 5)):
        x = rng.randrange(width)
        y = rng.randrange(height)

        rx = rng.randint(4, 25)
        ry = rng.randint(2, 12)

        draw.ellipse(
            (
                x - rx,
                y - ry,
                x + rx,
                y + ry
            ),
            fill=(
                80,
                65,
                45,
                rng.randint(3, 10)
            )
        )

    return image


def render_glyph(
    character,
    font_path,
    font_size,
    ink_color
):
    """
    رسم grapheme واحد بسرعة، بدون تدوير كل حرف.
    """

    font = ImageFont.truetype(
        str(font_path),
        font_size
    )

    bbox = font.getbbox(character)

    if bbox is None:
        return None

    left, top, right, bottom = bbox

    glyph_width = max(
        4,
        right - left + 10
    )

    glyph_height = max(
        4,
        bottom - top + 10
    )

    mask = Image.new(
        "L",
        (
            glyph_width,
            glyph_height
        ),
        0
    )

    ImageDraw.Draw(mask).text(
        (
            5 - left,
            5 - top
        ),
        character,
        font=font,
        fill=255
    )

    if mask.getbbox() is None:
        return None

    ink = Image.new(
        "RGBA",
        mask.size,
        ink_color + (0,)
    )

    ink.putalpha(mask)

    # تغيير بسيط في حجم الحرف
    scale = rng.uniform(
        0.92,
        1.08
    )

    if abs(scale - 1.0) > 0.02:
        ink = ink.resize(
            (
                max(
                    2,
                    int(ink.width * scale)
                ),
                max(
                    2,
                    int(ink.height * scale)
                )
            ),
            Image.Resampling.BILINEAR
        )

    advance = (
        font.getlength(character) * scale
        + rng.uniform(-2.0, 3.0)
    )

    return (
        ink,
        max(8.0, advance)
    )


def transform_image_and_boxes(
    image,
    boxes
):
    """
    تدوير الصورة وتحويل الصناديق بنفس التحويل.
    """

    array = np.asarray(
        image.convert("RGB")
    )

    height, width = array.shape[:2]

    angle = rng.uniform(
        -3.0,
        3.0
    )

    scale = rng.uniform(
        0.98,
        1.02
    )

    matrix = cv2.getRotationMatrix2D(
        (
            width / 2,
            height / 2
        ),
        angle,
        scale
    )

    matrix[0, 2] += (
        rng.uniform(-0.01, 0.01)
        * width
    )

    matrix[1, 2] += (
        rng.uniform(-0.01, 0.01)
        * height
    )

    transformed_array = cv2.warpAffine(
        array,
        matrix,
        (
            width,
            height
        ),
        flags=cv2.INTER_LINEAR,
        borderMode=cv2.BORDER_REFLECT_101
    )

    transformed_boxes = []

    for x1, y1, x2, y2 in boxes:
        points = np.array(
            [
                [x1, y1],
                [x2, y1],
                [x2, y2],
                [x1, y2],
            ],
            dtype=np.float32
        )

        new_points = cv2.transform(
            points[None],
            matrix
        )[0]

        minimum = new_points.min(
            axis=0
        )

        maximum = new_points.max(
            axis=0
        )

        new_x1 = max(
            0,
            min(
                width - 1,
                minimum[0]
            )
        )

        new_y1 = max(
            0,
            min(
                height - 1,
                minimum[1]
            )
        )

        new_x2 = max(
            new_x1 + 1,
            min(
                width,
                maximum[0]
            )
        )

        new_y2 = max(
            new_y1 + 1,
            min(
                height,
                maximum[1]
            )
        )

        transformed_boxes.append([
            float(new_x1),
            float(new_y1),
            float(new_x2),
            float(new_y2)
        ])

    return (
        Image.fromarray(transformed_array),
        transformed_boxes
    )


def make_raw_sample(index):
    """
    توليد صورة واحدة وملفاتها:
    JPG + YOLO TXT + JSON
    """

    width, height = IMAGE_SIZE

    image = paper_background(
        width,
        height
    ).convert("RGBA")

    text_layer = Image.new(
        "RGBA",
        (
            width,
            height
        ),
        (0, 0, 0, 0)
    )

    records = []

    number_of_lines = rng.randint(
        2,
        4
    )

    # ValueError: empty range in randrange
    min_line_gap = max(
        60,
        height // (number_of_lines + 2)
    )

    max_line_gap = min(
        120,
        height // number_of_lines
    )

    # ضمان أن الحد الأعلى ليس أصغر من الحد الأدنى
    max_line_gap = max(
        min_line_gap,
        max_line_gap
    )

    line_gap = rng.randint(
        min_line_gap,
        max_line_gap
    )

    top = rng.randint(
        18,
        42
    )

    for line_index in range(
        number_of_lines
    ):
        y = (
            top
            + line_index * line_gap
            + rng.randint(-6, 6)
        )

        # اختيار سطر كامل من القاموس
        source_line = graphemes(
            rng.choice(MODI_TEXTS)
        )

        # الاحتفاظ فقط بالحروف التي لها كلاس
        line = [
            grapheme
            for grapheme in source_line
            if (
                grapheme in CLASS_TO_ID
                or grapheme.isspace()
            )
        ]

        if not line:
            continue

        # تكرار السطر إذا كان قصيرًا
        while len(line) < 20:
            line += [" "] + line

        # تقليل عدد الحروف لتسريع التوليد
        line = line[
            :rng.randint(20, 55)
        ]

        x = rng.randint(
            22,
            62
        )

        base_font_size = rng.randint(
            50,
            74
        )

        for character in line:

            if character.isspace():
                x += rng.randint(
                    12,
                    25
                )
                continue

            # حماية إضافية من KeyError
            if character not in CLASS_TO_ID:
                continue

            rendered = render_glyph(
                character=character,
                font_path=rng.choice(FONT_FILES),
                font_size=max(
                    28,
                    int(
                        base_font_size
                        * rng.uniform(
                            0.92,
                            1.08
                        )
                    )
                ),
                ink_color=rng.choice([
                    (8, 8, 10),
                    (22, 21, 18),
                    (42, 40, 34),
                    (18, 28, 55),
                    (48, 30, 20),
                ])
            )

            if rendered is None:
                continue

            glyph, advance = rendered

            glyph_y = (
                y
                + rng.randint(-7, 7)
            )

            if x + glyph.width > width - 16:
                break

            text_layer.alpha_composite(
                glyph,
                (
                    int(x),
                    int(glyph_y)
                )
            )

            glyph_bbox = (
                glyph
                .getchannel("A")
                .getbbox()
            )

            if glyph_bbox is not None:
                gx1, gy1, gx2, gy2 = glyph_bbox

                records.append({
                    "text": character,
                    "class_id": CLASS_TO_ID[character],
                    "bbox": [
                        float(x + gx1),
                        float(glyph_y + gy1),
                        float(x + gx2),
                        float(glyph_y + gy2),
                    ],
                })

            x += max(
                15,
                advance + rng.uniform(-5, 5)
            )

    # إذا لم يتم رسم أي حرف
    if not records:
        return False

    # دمج الحروف مع الخلفية
    image = Image.alpha_composite(
        image,
        text_layer
    ).convert("RGB")

    array = np.asarray(
        image
    ).copy()

    # تغيير إضاءة سريع
    if rng.random() < 0.35:
        brightness = rng.uniform(
            0.90,
            1.08
        )

        array = np.clip(
            array.astype(np.float32)
            * brightness,
            0,
            255
        ).astype(np.uint8)

    image = Image.fromarray(
        array
    )

    original_boxes = [
        record["bbox"]
        for record in records
    ]

    image, transformed_boxes = (
        transform_image_and_boxes(
            image,
            original_boxes
        )
    )

    for record, box in zip(
        records,
        transformed_boxes
    ):
        record["bbox"] = box

    stem = f"modi_{index:06d}"

    image_path = (
        RAW_IMAGES / f"{stem}.jpg"
    )

    label_path = (
        RAW_LABELS / f"{stem}.txt"
    )

    annotation_path = (
        RAW_LABELS / f"{stem}.json"
    )

    image.save(
        image_path,
        quality=rng.randint(
            90,
            95
        ),
        subsampling=0
    )

    # إنشاء YOLO labels
    label_lines = []

    for record in records:
        x1, y1, x2, y2 = record["bbox"]
        class_id = record["class_id"]

        label_lines.append(
            f"{class_id} "
            f"{((x1 + x2) / 2) / width:.6f} "
            f"{((y1 + y2) / 2) / height:.6f} "
            f"{(x2 - x1) / width:.6f} "
            f"{(y2 - y1) / height:.6f}"
        )

    label_path.write_text(
        "\n".join(label_lines) + "\n",
        encoding="utf-8"
    )

    # JSON للمراجعة
    annotation_path.write_text(
        json.dumps(
            {
                "image": stem,
                "width": width,
                "height": height,
                "text": "".join(
                    record["text"]
                    for record in records
                ),
                "annotations": records,
            },
            ensure_ascii=False,
            indent=2
        ),
        encoding="utf-8"
    )

    return True


# حذف البيانات القديمة
if OVERWRITE:
    for path in RAW_IMAGES.glob("*.jpg"):
        path.unlink()

    for path in RAW_LABELS.glob("*.txt"):
        path.unlink()

    for path in RAW_LABELS.glob("*.json"):
        path.unlink()


# توليد العدد المطلوب من الصور
generated = 0
index = 0

existing = {
    path.stem
    for path in RAW_IMAGES.glob("modi_*.jpg")
}

with tqdm(
    total=N_SAMPLES,
    desc="Generating Modi samples"
) as progress:

    while generated < N_SAMPLES:
        stem = f"modi_{index:06d}"

        if stem not in existing:
            success = make_raw_sample(index)

            if success:
                generated += 1
                progress.update(1)

        index += 1

print(
    "عدد الصور الخام:",
    len(list(RAW_IMAGES.glob("*.jpg")))
)


Generating Modi samples:   0%|          | 0/5000 [00:00<?, ?it/s]

عدد الصور الخام: 5000


In [ ]:
# الخلية 6 — تحويل الخام إلى YOLO وتقسيم train/val/test
from sklearn.model_selection import train_test_split
pairs=[]
for jp in sorted(RAW_LABELS.glob('*.json')):
    item=json.loads(jp.read_text(encoding='utf-8')); ip=RAW_IMAGES/(item['image']+'.jpg')
    if ip.exists() and item.get('annotations'): pairs.append((ip,jp))
if len(pairs)<30: raise RuntimeError('عدد الصور قليل جدًا للتقسيم.')
for split in ['train','val','test']:
    for p in (DATASET/f'images/{split}').glob('*'): p.unlink()
    for p in (DATASET/f'labels/{split}').glob('*'): p.unlink()
train,rest=train_test_split(pairs,test_size=.20,random_state=SEED); val,test=train_test_split(rest,test_size=.50,random_state=SEED)
def convert(split, rows):
    for ip,jp in rows:
        shutil.copy2(ip,DATASET/f'images/{split}'/ip.name); item=json.loads(jp.read_text(encoding='utf-8')); W,H=item['width'],item['height']; out=[]
        for a in item['annotations']:
            x1,y1,x2,y2=a['bbox']; x1=max(0,min(W-1,x1)); y1=max(0,min(H-1,y1)); x2=max(x1+1,min(W,x2)); y2=max(y1+1,min(H,y2))
            out.append(f"{a['class_id']} {((x1+x2)/2)/W:.6f} {((y1+y2)/2)/H:.6f} {(x2-x1)/W:.6f} {(y2-y1)/H:.6f}")
        (DATASET/f'labels/{split}'/(ip.stem+'.txt')).write_text('\n'.join(out)+'\n',encoding='utf-8')
for name,rows in [('train',train),('val',val),('test',test)]: convert(name,rows); print(name,len(rows))
names='\n'.join(f'  {i}: {json.dumps(ch,ensure_ascii=False)}' for ch,i in CLASS_TO_ID.items())
YAML=DATASET/'modi.yaml'; YAML.write_text(f'path: {DATASET}\ntrain: images/train\nval: images/val\ntest: images/test\nnames:\n{names}\n',encoding='utf-8')
print('YAML:',YAML)


train 4000
val 500
test 500
YAML: /content/drive/MyDrive/modi_ocr_yolo11s_project/dataset/modi.yaml


In [ ]:
# الخلية 7 — فحص GPU والكاش المحلي
import torch
HAS_CUDA=bool(torch.cuda.is_available()); DEVICE=0 if HAS_CUDA else 'cpu'; AMP=HAS_CUDA; BATCH=4 if HAS_CUDA else 1
LOCAL_ROOT=Path('/content/modi_ocr_local_cache'); LOCAL_DATASET=LOCAL_ROOT/'dataset'; LOCAL_RUNS=LOCAL_ROOT/'runs'
if LOCAL_ROOT.exists(): shutil.rmtree(LOCAL_ROOT)
LOCAL_ROOT.mkdir(parents=True,exist_ok=True); shutil.copytree(DATASET,LOCAL_DATASET); LOCAL_RUNS.mkdir(parents=True,exist_ok=True)
LOCAL_YAML=LOCAL_DATASET/'modi.yaml'; LOCAL_YAML.write_text(LOCAL_YAML.read_text(encoding='utf-8').replace(str(DATASET),str(LOCAL_DATASET)),encoding='utf-8')
print('CUDA:',HAS_CUDA,'| GPU:',torch.cuda.get_device_name(0) if HAS_CUDA else 'CPU')


CUDA: True | GPU: Tesla T4


In [ ]:
# @title
# الخلية 8 — تدريب YOLO11s بنفس نسق الدفتر الثاني
from ultralytics import YOLO
model=YOLO('yolo11s.pt')
model.train(data=str(LOCAL_YAML),epochs=45,imgsz=512,batch=BATCH,device=DEVICE,workers=1,patience=12,cache='disk',amp=AMP,pretrained=True,optimizer='AdamW',lr0=.001,lrf=.01,cos_lr=True,degrees=2,translate=.03,scale=.15,shear=1,fliplr=0.0,flipud=0.0,mosaic=.20,mixup=0.0,hsv_h=0.0,hsv_s=0.0,hsv_v=.15,project=str(LOCAL_RUNS),name='modi_yolo_multiclass',exist_ok=True)
BEST=LOCAL_RUNS/'modi_yolo_multiclass'/'weights'/'best.pt'
if not BEST.exists(): raise FileNotFoundError(BEST)
PERSISTENT_BEST=RUNS/'modi_yolo_multiclass'/'weights'/'best.pt'; PERSISTENT_BEST.parent.mkdir(parents=True,exist_ok=True); shutil.copy2(BEST,PERSISTENT_BEST)
print('أفضل نموذج:',PERSISTENT_BEST)


In [ ]:
# الخلية 8 — تدريب YOLO11s مع كامل احتياطات الذاكرة واستئناف التدريب الآلي
import os
import gc
import random
import shutil
from pathlib import Path

# تقليل تجزئة ذاكرة CUDA في بيئة Colab
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')

import torch
from ultralytics import YOLO, __version__ as ULTRALYTICS_VERSION

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)
if HAS_CUDA:
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = True

# مسارات النموذج والنقاط المرجعية
RUN_NAME = 'modi_yolo_multiclass'
MODEL_RUN_DIR = LOCAL_RUNS / RUN_NAME
WEIGHTS_DIR = MODEL_RUN_DIR / 'weights'
LAST_CHECKPOINT = WEIGHTS_DIR / 'last.pt'
BEST_CHECKPOINT = WEIGHTS_DIR / 'best.pt'
PERSISTENT_BEST = RUNS / RUN_NAME / 'weights' / 'best.pt'

PRETRAINED_MODEL = 'yolo11s.pt'

def clear_memory():
    """تحرير ذاكرة Python وCUDA تماماً."""
    gc.collect()
    if HAS_CUDA:
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()

def gpu_memory_gb():
    """حساب الذاكرة المتاحة في كارت الشاشة."""
    if not HAS_CUDA:
        return None
    free_bytes, total_bytes = torch.cuda.mem_get_info()
    return free_bytes / 1024**3, total_bytes / 1024**3

def safe_batch_size(default=16):
    """تحديد BATCH مناسب تلقائياً بناءً على حجم الذاكرة الحرة."""
    if not HAS_CUDA:
        return 1
    free_gb, total_gb = gpu_memory_gb()
    if free_gb < 5:
        batch = 2
    elif free_gb < 8:
        batch = 4
    elif free_gb < 12:
        batch = 8
    else:
        batch = default
    print(f"GPU: {torch.cuda.get_device_name(0)} | free={free_gb:.2f}GB / total={total_gb:.2f}GB | Selected Batch={batch}")
    return batch

def train_once(batch_size, resume=True):
    clear_memory()

    # اختيار نقطة البداية: استئناف من last.pt إن وجد أو البدء من النماذج المدربة مسبقاً
    checkpoint = str(LAST_CHECKPOINT) if resume and LAST_CHECKPOINT.exists() else PRETRAINED_MODEL
    model = YOLO(checkpoint)

    print(f"نقطة البداية: {checkpoint}")
    print(f"إصدار Ultralytics: {ULTRALYTICS_VERSION}")

    model.train(
        data=str(LOCAL_YAML),
        epochs=45,
        imgsz=512,
        batch=batch_size,
        device=DEVICE,
        workers=2,
        cache='disk',
        amp=AMP,
        pretrained=True,
        resume=resume,
        optimizer='AdamW',
        lr0=0.001,
        lrf=0.01,
        cos_lr=True,
        patience=12,
        save=True,
        save_period=5,
        degrees=2,
        translate=0.03,
        scale=0.15,
        shear=1,
        fliplr=0.0,
        flipud=0.0,
        mosaic=0.20,
        mixup=0.0,
        hsv_h=0.0,
        hsv_s=0.0,
        hsv_v=0.15,
        project=str(LOCAL_RUNS),
        name=RUN_NAME,
        exist_ok=True,
        seed=SEED,
        verbose=True
    )
    return model

# تحديد الـ Batch الأولي تلقائياً والتحقق من وجود نقطة حفظ سابقة
initial_batch = safe_batch_size(default=16)
resume_training = LAST_CHECKPOINT.exists()

try:
    model = train_once(initial_batch, resume=resume_training)
except RuntimeError as error:
    error_text = str(error).lower()
    is_oom = "out of memory" in error_text or "cuda error" in error_text
    if not is_oom:
        raise

    print(" تم اكتشاف امتلاء ذاكرة GPU (Out of Memory)! جارٍ التنظيف وإعادة المحاولة بـ Batch أصغر...")
    clear_memory()
    retry_batch = max(1, initial_batch // 2)
    model = train_once(retry_batch, resume=False)

# حفظ النموذج الأفضل على Google Drive لتفادي ضياعه
if BEST_CHECKPOINT.exists():
    PERSISTENT_BEST.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(BEST_CHECKPOINT, PERSISTENT_BEST)
    print(f" تم حفظ أفضل نموذج بنجاح في: {PERSISTENT_BEST}")
else:
    raise FileNotFoundError(f"لم يتم العثور على النموذج الأفضل في {BEST_CHECKPOINT}")

clear_memory()
print(" اكتمل التدريب ونُظفت الذاكرة بنجاح.")

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart#ultralytics-settings.
GPU: Tesla T4 | free=14.46GB / total=14.56GB | Selected Batch=16
نقطة البداية: yolo11s.pt
إصدار Ultralytics: 8.4.143
Ultralytics 8.4.143 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=disk, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/content/modi_ocr_local_cache/dataset/modi.yaml, degrees=2, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, 

/usr/local/lib/python3.13/dist-packages/ultralytics/utils/metrics.py:607: UserWarning: Glyph 71168 (\N{MODI LETTER A}) missing from font(s) Arial Unicode MS.
  fig.savefig(plot_fname, dpi=250)
/usr/local/lib/python3.13/dist-packages/ultralytics/utils/metrics.py:607: UserWarning: Glyph 71169 (\N{MODI LETTER AA}) missing from font(s) Arial Unicode MS.
  fig.savefig(plot_fname, dpi=250)
/usr/local/lib/python3.13/dist-packages/ultralytics/utils/metrics.py:607: UserWarning: Glyph 71171 (\N{MODI LETTER II}) missing from font(s) Arial Unicode MS.
  fig.savefig(plot_fname, dpi=250)
/usr/local/lib/python3.13/dist-packages/ultralytics/utils/metrics.py:607: UserWarning: Glyph 71172 (\N{MODI LETTER U}) missing from font(s) Arial Unicode MS.
  fig.savefig(plot_fname, dpi=250)
/usr/local/lib/python3.13/dist-packages/ultralytics/utils/metrics.py:607: UserWarning: Glyph 71178 (\N{MODI LETTER E}) missing from font(s) Arial Unicode MS.
  fig.savefig(plot_fname, dpi=250)
/usr/local/lib/python3.13/dist-pa

                   all        500      37622      0.999      0.999      0.995      0.962
                     𑘀        422       1864          1          1      0.995      0.974
                     𑘁        474       2746          1          1      0.995      0.972
                     𑘃        277        703          1          1      0.995      0.965
                     𑘄        331       1016          1          1      0.995      0.977
                     𑘊        197        491      0.999          1      0.995      0.977
                     𑘋         49         81      0.998          1      0.995      0.979
                     𑘌         77        149      0.999          1      0.995      0.973
                     𑘍         58        113      0.996          1      0.995      0.977
                     𑘎        456       2286          1      0.999      0.995      0.963
                     𑘏        186        396      0.998          1      0.995      0.974
                     

In [ ]:
# الخلية 9 — التقييم والتنبؤ على مجموعة الاختبار والحفظ في Google Drive
import shutil
from ultralytics import YOLO

# 1. تحميل أفضل نموذج تم حفظه أثناء التدريب
if not BEST_CHECKPOINT.exists():
    raise FileNotFoundError(f"لم يتم العثور على أفضل نموذج في: {BEST_CHECKPOINT}")

trained = YOLO(str(BEST_CHECKPOINT))

# 2. تقييم النموذج على مجموعة الاختبار (test split)
metrics = trained.val(
    data=str(LOCAL_YAML),
    split='test',
    imgsz=512,
    conf=0.25,
    iou=0.50,
    device=DEVICE
)

print('\n' + '='*30)
print('📊 نتائج التقييم على مجموعة الاختبار:')
print('mAP50    =', float(metrics.box.map50))
print('mAP50-95 =', float(metrics.box.map))
print('='*30 + '\n')

# 3. التنبؤ على صور الاختبار وحفظ التسميات والتوضيحات بصرياً
PRED_DIR_LOCAL = LOCAL_RUNS / 'test_predictions'

trained.predict(
    source=str(LOCAL_DATASET / 'images/test'),
    imgsz=512,
    conf=0.25,
    save=True,      # حفظ الصور المعلمة بالمربعات
    save_txt=True,  # حفظ ملفات TXT التي تحتوي إحداثيات التنبؤ
    project=str(LOCAL_RUNS),
    name='test_predictions',
    exist_ok=True
)

# 4. نقل نتائج التنبؤات والتقييم بالكامل إلى Google Drive الحقيقي
PRED_DIR_DRIVE = RUNS / 'test_predictions'
if PRED_DIR_LOCAL.exists():
    shutil.copytree(PRED_DIR_LOCAL, PRED_DIR_DRIVE, dirs_exist_ok=True)
    print(f"✅ تم حفظ جميع التنبؤات ونتائج الاختبار بنجاح في Google Drive:\n {PRED_DIR_DRIVE}")

Ultralytics 8.4.143 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLO11s summary (fused): 100 layers, 9,428,280 parameters, 0 gradients, 21.5 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 66.0±8.7 MB/s, size: 145.8 KB)
val: Scanning /content/modi_ocr_local_cache/dataset/labels/test... 500 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 500/500 452.5it/s 1.1s
val: New cache created: /content/modi_ocr_local_cache/dataset/labels/test.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 32/32 2.7it/s 12.0s


/usr/local/lib/python3.13/dist-packages/ultralytics/utils/metrics.py:607: UserWarning: Glyph 71168 (\N{MODI LETTER A}) missing from font(s) Arial Unicode MS.
  fig.savefig(plot_fname, dpi=250)
/usr/local/lib/python3.13/dist-packages/ultralytics/utils/metrics.py:607: UserWarning: Glyph 71169 (\N{MODI LETTER AA}) missing from font(s) Arial Unicode MS.
  fig.savefig(plot_fname, dpi=250)
/usr/local/lib/python3.13/dist-packages/ultralytics/utils/metrics.py:607: UserWarning: Glyph 71171 (\N{MODI LETTER II}) missing from font(s) Arial Unicode MS.
  fig.savefig(plot_fname, dpi=250)
/usr/local/lib/python3.13/dist-packages/ultralytics/utils/metrics.py:607: UserWarning: Glyph 71172 (\N{MODI LETTER U}) missing from font(s) Arial Unicode MS.
  fig.savefig(plot_fname, dpi=250)
/usr/local/lib/python3.13/dist-packages/ultralytics/utils/metrics.py:607: UserWarning: Glyph 71178 (\N{MODI LETTER E}) missing from font(s) Arial Unicode MS.
  fig.savefig(plot_fname, dpi=250)
/usr/local/lib/python3.13/dist-pa

                   all        500      37551      0.999          1      0.995      0.971
                     𑘀        431       1791          1          1      0.995      0.982
                     𑘁        473       2712          1          1      0.995       0.98
                     𑘃        309        805          1          1      0.995      0.977
                     𑘄        345       1074      0.999          1      0.995      0.987
                     𑘊        208        504          1          1      0.995       0.98
                     𑘋         53         94          1          1      0.995      0.982
                     𑘌         83        167          1          1      0.995       0.99
                     𑘍         66        115          1          1      0.995      0.982
                     𑘎        452       2142          1          1      0.995      0.974
                     𑘏        168        370      0.997          1      0.995      0.978
                     

In [ ]:
# @title
# الخلية 10 — ترتيب الصناديق وإخراج النص الحقيقي + Gradio
import gradio as gr

def sort_character_boxes(items):
    if not items: return []
    threshold=max(10.0,.55*float(np.median([b[3]-b[1] for b in items]))); lines=[]
    for item in items:
        cy=(item[1]+item[3])/2; target=None
        for line in lines:
            if abs(cy-line['cy'])<=threshold: target=line; break
        if target is None: lines.append({'cy':cy,'items':[item]})
        else: target['items'].append(item); target['cy']=float(np.mean([(z[1]+z[3])/2 for z in target['items']]))
    lines.sort(key=lambda z:z['cy']); return [sorted(z['items'],key=lambda a:a[0]) for z in lines]

def predict_image(image, conf=0.25):
    if image is None: return None,'',''
    pil=image.convert('RGB') if isinstance(image,Image.Image) else Image.fromarray(image).convert('RGB'); result=trained.predict(source=pil,imgsz=640,conf=float(conf),verbose=False)[0]
    out=pil.copy(); draw=ImageDraw.Draw(out); found=[]
    for box,cls,score in zip(result.boxes.xyxy.cpu().numpy(),result.boxes.cls.cpu().numpy(),result.boxes.conf.cpu().numpy()):
        x1,y1,x2,y2=map(float,box); cid=int(cls); score=float(score)
        if cid in ID_TO_CLASS: found.append((x1,y1,x2,y2,ID_TO_CLASS[cid],score))
    ordered=sort_character_boxes(found)
    for line in ordered:
        for x1,y1,x2,y2,ch,score in line:
            draw.rectangle((x1,y1,x2,y2),outline='red',width=max(2,out.width//400)); draw.text((x1,max(0,y1-25)),f'{ch} {score:.2f}',fill='red')
    text='\n'.join(''.join(z[4] for z in line) for line in ordered)
    details='\n'.join(f'{i+1}. {z[4]} | confidence={z[5]:.3f} | box={tuple(round(v,1) for v in z[:4])}' for i,line in enumerate(ordered) for z in line)
    return out,text,details

demo=gr.Interface(fn=predict_image,inputs=[gr.Image(type='pil',label='صورة Modi'),gr.Slider(.05,.90,value=.25,step=.05,label='Confidence')],outputs=[gr.Image(label='الكشوف'),gr.Textbox(label='النص المستخرج',lines=10),gr.Textbox(label='تفاصيل الصناديق',lines=10)],title='Modi OCR — YOLO11s Multi-class')
demo.launch(share=True)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://4d64c9b5326dcc2c29.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
!apt-get install -y fonts-noto-hinted fonts-noto-unhinted

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  fonts-noto-core fonts-noto-ui-core
The following NEW packages will be installed:
  fonts-noto-core fonts-noto-hinted fonts-noto-ui-core fonts-noto-unhinted
0 upgraded, 4 newly installed, 0 to remove and 57 not upgraded.
Need to get 13.6 MB of archives.
After this operation, 48.8 MB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 fonts-noto-core all 20201225-1build1 [12.2 MB]
Get:2 http://archive.ubuntu.com/ubuntu jammy/main amd64 fonts-noto-ui-core all 20201225-1build1 [1,420 kB]
Get:3 http://archive.ubuntu.com/ubuntu jammy/universe amd64 fonts-noto-hinted all 20201225-1build1 [3,988 B]
Get:4 http://archive.ubuntu.com/ubuntu jammy/universe amd64 fonts-noto-unhinted all 20201225-1build1 [16.8 kB]
Fetched 13.6 MB in 1s (11.5 MB/s)
Selecting previously unselected package fonts-noto-core.
(Read

In [ ]:
# @title
import gradio as gr
import torch
import numpy as np
from PIL import Image

# تحميل النموذج
model = YOLO(str(BEST_CHECKPOINT))

def process_modi_ocr(image, conf_threshold):
    if image is None:
        return None, ""

    # 1. التنبؤ بتقليل iou للسماح بكشف الأحرف المتلاصقة
    results = model.predict(
        source=image,
        conf=conf_threshold,
        iou=0.30,  # تقليل العتبة لعدم حذف الأحرف المتداخلة
        imgsz=512,
        device=DEVICE
    )[0]

    # رسم المربعات على الصورة
    annotated_img = Image.fromarray(results.plot()[:, :, ::-1])

    # 2. استخراج المربعات وترتيبها أفقياً ورأسياً
    boxes = []
    for box in results.boxes:
        x1, y1, x2, y2 = box.xyxy[0].tolist()
        cls_id = int(box.cls[0])
        # إيجاد الحرف من القاموس
        char = ID_TO_CLASS.get(cls_id, '')
        boxes.append({'char': char, 'x': x1, 'y': y1})

    if not boxes:
        return annotated_img, "لم يتم التعرف على أي حرف."

    # 3. فرز مكاني دقيق (أسطر ثم حروف من اليسار لليمين)
    # تجميع الأحرف في أسطر بناءً على المسافة الراسية (y)
    boxes.sort(key=lambda b: b['y'])

    lines = []
    current_line = [boxes[0]]
    line_threshold = 25  # المسافة بالبكسل للتمييز بين السطر والآخر

    for b in boxes[1:]:
        if abs(b['y'] - current_line[0]['y']) < line_threshold:
            current_line.append(b)
        else:
            # ترتيب الحروف داخل السطر من اليسار لليمين
            current_line.sort(key=lambda item: item['x'])
            lines.append("".join([item['char'] for item in current_line]))
            current_line = [b]

    if current_line:
        current_line.sort(key=lambda item: item['x'])
        lines.append("".join([item['char'] for item in current_line]))

    extracted_text = "\n".join(lines)

    return annotated_img, extracted_text

# بناء واجهة Gradio مع دعم التضمين
with gr.Blocks(css=".output-text { font-family: 'Noto Sans Modi', sans-serif; font-size: 22px; }") as demo:
    gr.Markdown("## Modi OCR — YOLO11s (نظام كشف وتجميع النصوص)")

    with gr.Row():
        with gr.Column():
            input_img = gr.Image(type="pil", label="صورة Modi")
            conf_slider = gr.Slider(minimum=0.05, maximum=0.90, value=0.15, step=0.05, label="Confidence Threshold")
            submit_btn = gr.Button("Submit", variant="primary")

        with gr.Column():
            output_img = gr.Image(label="الكشوف")
            # إضافة صنف CSS لعرض الخط بشكل صحيح
            output_text = gr.Textbox(label="النص المستخرج", lines=6, elem_classes=["output-text"])

    submit_btn.click(fn=process_modi_ocr, inputs=[input_img, conf_slider], outputs=[output_img, output_text])

demo.launch(share=True, debug=True)

/tmp/ipykernel_566/4076479664.py:63: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: css. Please pass these parameters to launch() instead.
  with gr.Blocks(css=".output-text { font-family: 'Noto Sans Modi', sans-serif; font-size: 22px; }") as demo:


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://062aa0e11d718cafda.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)



0: 192x512 1 𑘀, 4 𑘃s, 1 𑘄, 1 𑘊, 1 𑘋, 2 𑘎s, 1 𑘏, 1 𑘐, 1 𑘑, 1 𑘓, 26 𑘕s, 6 𑘖s, 2 𑘘s, 1 𑘚, 6 𑘛s, 5 𑘝s, 16 𑘡s, 1 𑘦, 9 𑘩s, 1 𑘬, 13.1ms
Speed: 1.5ms preprocess, 13.1ms inference, 1.4ms postprocess per image at shape (1, 3, 192, 512)

0: 192x512 1 𑘀, 10 𑘃s, 9 𑘄s, 5 𑘊s, 1 𑘋, 4 𑘎s, 2 𑘏s, 2 𑘐s, 1 𑘑, 3 𑘓s, 34 𑘕s, 8 𑘖s, 2 𑘗s, 2 𑘘s, 3 𑘚s, 13 𑘛s, 9 𑘝s, 2 𑘞s, 26 𑘡s, 3 𑘦s, 1 𑘧, 1 𑘨, 12 𑘩s, 2 𑘬s, 13.3ms
Speed: 1.6ms preprocess, 13.3ms inference, 2.1ms postprocess per image at shape (1, 3, 192, 512)
Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://4d64c9b5326dcc2c29.gradio.live
Killing tunnel 127.0.0.1:7861 <> https://062aa0e11d718cafda.gradio.live


In [ ]:
# تنزيل خط Noto Sans Modi مباشرة لاستخدامه في الرسم والواجهة
!wget -q https://github.com/googlefonts/noto-fonts/raw/main/hinted/ttf/NotoSansModi/NotoSansModi-Regular.ttf -O /content/NotoSansModi-Regular.ttf
print("✅ تم تحميل ملف الخط بنجاح.")

✅ تم تحميل ملف الخط بنجاح.


In [ ]:
# @title
import gradio as gr
import torch
import numpy as np
from PIL import Image, ImageDraw, ImageFont
from pathlib import Path

# تحميل النموذج
model = YOLO(str(BEST_CHECKPOINT))

# مسار الخط الذي تم تنزيله
FONT_PATH = "/content/NotoSansModi-Regular.ttf"

def process_modi_ocr(image, conf_threshold):
    if image is None:
        return None, ""

    # 1. التنبؤ بالأحرف
    results = model.predict(
        source=image,
        conf=conf_threshold,
        iou=0.30,
        imgsz=512,
        device=DEVICE
    )[0]

    # 2. استخراج المربعات والحروف
    boxes = []
    for box in results.boxes:
        x1, y1, x2, y2 = box.xyxy[0].tolist()
        cls_id = int(box.cls[0])
        char = ID_TO_CLASS.get(cls_id, '')
        boxes.append({'char': char, 'x': x1, 'y': y1, 'bbox': (x1, y1, x2, y2)})

    if not boxes:
        return image, "لم يتم التعرف على أي حرف."

    # 3. رسم المربعات والنصوص يدويًا باستخدام الخط المحمل لتفادي المربعات بصرياً
    annotated_img = image.copy().convert("RGB")
    draw = ImageDraw.Draw(annotated_img)

    try:
        font = ImageFont.truetype(FONT_PATH, 20)
    except Exception:
        font = ImageFont.load_default()

    for b in boxes:
        x1, y1, x2, y2 = b['bbox']
        draw.rectangle([x1, y1, x2, y2], outline="red", width=2)
        draw.text((x1, max(0, y1 - 22)), b['char'], fill="blue", font=font)

    # 4. الترتيب المكاني للأحرف (أسطر ثم حروف من اليسار لليمين)
    boxes.sort(key=lambda item: item['y'])

    lines = []
    current_line = [boxes[0]]
    line_threshold = 25  # المسافة بالبكسل لتحديد السطر الجديد

    for b in boxes[1:]:
        if abs(b['y'] - current_line[0]['y']) < line_threshold:
            current_line.append(b)
        else:
            current_line.sort(key=lambda item: item['x'])
            lines.append("".join([item['char'] for item in current_line]))
            current_line = [b]

    if current_line:
        current_line.sort(key=lambda item: item['x'])
        lines.append("".join([item['char'] for item in current_line]))

    extracted_text = "\n".join(lines)

    return annotated_img, extracted_text

# HTML Custom Font Injection لتجبار المتصفح على عرض خط Modi في حقل النص
custom_css = """
@font-face {
    font-family: 'NotoSansModi';
    src: url('https://raw.githubusercontent.com/googlefonts/noto-fonts/main/hinted/ttf/NotoSansModi/NotoSansModi-Regular.ttf');
}
textarea {
    font-family: 'NotoSansModi', sans-serif !important;
    font-size: 24px !important;
}
"""

with gr.Blocks(css=custom_css) as demo:
    gr.Markdown("## Modi OCR — YOLO11s (استخراج النص المباشر)")

    with gr.Row():
        with gr.Column():
            input_img = gr.Image(type="pil", label="صورة Modi")
            conf_slider = gr.Slider(minimum=0.05, maximum=0.90, value=0.15, step=0.05, label="Confidence Threshold")
            submit_btn = gr.Button("تطبيق الـ OCR", variant="primary")

        with gr.Column():
            output_img = gr.Image(label="الكشوف المحددة")
            output_text = gr.Textbox(label="النص المستخرج (Modi Text)", lines=6)

    submit_btn.click(fn=process_modi_ocr, inputs=[input_img, conf_slider], outputs=[output_img, output_text])

demo.launch(share=True, debug=True)

/tmp/ipykernel_566/3669191413.py:86: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: css. Please pass these parameters to launch() instead.
  with gr.Blocks(css=custom_css) as demo:


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://e0c971bbfee7980bcf.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)



0: 192x512 1 𑘀, 5 𑘃s, 6 𑘄s, 2 𑘊s, 1 𑘌, 15 𑘎s, 1 𑘏, 2 𑘐s, 1 𑘑, 2 𑘓s, 35 𑘕s, 3 𑘖s, 1 𑘗, 3 𑘛s, 3 𑘜s, 1 𑘝, 2 𑘠s, 5 𑘡s, 4 𑘢s, 9 𑘣s, 4 𑘥s, 3 𑘦s, 1 𑘨, 1 𑘩, 1 𑘬, 10.4ms
Speed: 1.5ms preprocess, 10.4ms inference, 1.7ms postprocess per image at shape (1, 3, 192, 512)
Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7861 <> https://e0c971bbfee7980bcf.gradio.live


In [ ]:
# @title
# الخلية 9 — تصدير النموذج إلى ONNX وحفظ قاموس الفئات (classes.json)
import json
import shutil
from pathlib import Path
from ultralytics import YOLO

# 1. تحميل النموذج الأفضل (إما من التخزين الدائم أو المحلي)
model_path = PERSISTENT_BEST if 'PERSISTENT_BEST' in globals() and PERSISTENT_BEST.exists() else BEST_CHECKPOINT

if not Path(model_path).exists():
    raise FileNotFoundError(f"لم يتم العثور على ملف النموذج في: {model_path}")

print(f"جارٍ تحميل النموذج للتصدير من: {model_path}")
export_model = YOLO(str(model_path))

# 2. تصدير النموذج إلى صيغة ONNX
print("جارٍ تحويل النموذج إلى صيغة ONNX...")
onnx_exported_path = export_model.export(
    format='onnx',
    imgsz=512,        # الدقة المطابقة لعملية التدريب (يمكن تغييرها إلى 640 حسب الحاجة)
    opset=12,
    simplify=True,
    dynamic=False,
    nms=False
)

onnx_file = Path(onnx_exported_path)

# 3. التأكد من إنشاء مجلد EXPORTS وتفريغه
EXPORTS.mkdir(parents=True, exist_ok=True)

# 4. نقل ملف ONNX إلى مجلد EXPORTS في Google Drive
dest_onnx = EXPORTS / 'modi_yolo_multiclass.onnx'
shutil.copy2(onnx_file, dest_onnx)

# 5. حفظ أسماء الفئات والقاموس في ملف classes.json للاستخدام في التطبيقات
classes_json_path = EXPORTS / 'classes.json'

# جلب الفئات من النموذج أو من CLASS_TO_ID
if hasattr(export_model, 'names') and export_model.names:
    classes_data = export_model.names
elif 'CLASS_TO_ID' in globals():
    # عكس القاموس للربط بين ID والحرف
    classes_data = {v: k for k, v in CLASS_TO_ID.items()}
else:
    classes_data = {}

classes_json_path.write_text(
    json.dumps(classes_data, ensure_ascii=False, indent=2),
    encoding='utf-8'
)

print("\n" + "="*40)
print("✅ تم التصدير وتجهيز الملفات بنجاح!")
print('ملف النموذج (ONNX):', dest_onnx)
print('ملف الفئات (Classes):', classes_json_path)
print("="*40)

جارٍ تحميل النموذج للتصدير من: /content/drive/MyDrive/modi_ocr_yolo11s_project/runs/modi_yolo_multiclass/weights/best.pt
جارٍ تحويل النموذج إلى صيغة ONNX...
Ultralytics 8.4.143 🚀 Python-3.13.15 torch-2.11.0+cu128 CPU (Intel Xeon CPU @ 2.00GHz)
WARNING ⚠️ This model has no one-to-one head; using one-to-many outputs.
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino
YOLO11s summary (fused): 100 layers, 9,428,280 parameters, 0 gradients, 13.7 GFLOPs

PyTorch: starting from '/content/drive/MyDrive/modi_ocr_yolo11s_project/runs/modi_yolo_multiclass/weights/best.pt' with input shape (1, 3, 512, 512) BCHW and output shape(s) (1, 44, 5376) (18.3 MB)
requirements: Ultralytics requirement ['onnxslim>=0.1.82'] not found, attempting AutoUpdate...
Using Python 3.13.15 environment at: /usr
Resolved 10 packages in 760ms
Prepared 2 packages in 61ms
Installed 2 packages in 11ms
 + colorama==0.4.6
 + onnxslim==0.1

In [ ]:
import shutil
from pathlib import Path
from ultralytics import YOLO

# 1. تحديد المسارات المدخلة
SOURCE_IMAGES_DIR = Path('/content/drive/MyDrive/Data/Real/images')
SOURCE_TEXT_LABELS_DIR = Path('/content/drive/MyDrive/Data/Real/labels')

# 2. تحديد مسار المخرج الرئيسي وإنشاء المجلدات الفرعية الـ 3
DEST_BASE = Path('/content/drive/MyDrive/Data/Data_Labled')
DEST_IMAGES = DEST_BASE / 'images'          # مجلد الصور (200 صورة)
DEST_TEXT_LABELS = DEST_BASE / 'text_labels' # مجلد النصوص الأصلية
DEST_BOX_LABELS = DEST_BASE / 'box_labels'   # مجلد إحداثيات البوكسات (YOLO .txt)

for d in [DEST_IMAGES, DEST_TEXT_LABELS, DEST_BOX_LABELS]:
    d.mkdir(parents=True, exist_ok=True)

# 3. تحميل نموذج YOLO المدرب
model_path = PERSISTENT_BEST if 'PERSISTENT_BEST' in globals() and PERSISTENT_BEST.exists() else BEST_CHECKPOINT
model = YOLO(str(model_path))

# 4. جلب جميع الصور وأخذ أول 200 صورة فقط
image_extensions = ('*.jpg', '*.jpeg', '*.png', '*.bmp')
all_images = [p for ext in image_extensions for p in SOURCE_IMAGES_DIR.glob(ext)]
selected_images = all_images[:200]  # تحديد 200 صورة فقط

print(f"📁 تم إيجاد {len(all_images)} صورة. جاري معالجة ونقل {len(selected_images)} صورة فقط...\n")

count = 0
for img_path in selected_images:
    count += 1
    file_stem = img_path.stem

    # أ) نسخ الصورة إلى مجلد الصور الجديد
    shutil.copy2(img_path, DEST_IMAGES / img_path.name)

    # ب) نقل/نسخ ملف النص الأصلي (إن وجد)
    text_label_file = SOURCE_TEXT_LABELS_DIR / f"{file_stem}.txt"
    if text_label_file.exists():
        shutil.copy2(text_label_file, DEST_TEXT_LABELS / f"{file_stem}.txt")

    # ج) التنبؤ وتوليد إحداثيات البوكسات (YOLO Box Labels)
    results = model.predict(
        source=str(img_path),
        conf=0.15,
        iou=0.30,
        imgsz=512,
        device=DEVICE,
        verbose=False
    )[0]

    img_h, img_w = results.orig_shape
    yolo_lines = []

    for box in results.boxes:
        cls_id = int(box.cls[0])
        x1, y1, x2, y2 = box.xyxy[0].tolist()

        x_center = ((x1 + x2) / 2.0) / img_w
        y_center = ((y1 + y2) / 2.0) / img_h
        width = (x2 - x1) / img_w
        height = (y2 - y1) / img_h

        yolo_lines.append(f"{cls_id} {x_center:.6f} {y_center:.6f} {width:.6f} {height:.6f}")

    # حفظ ملف إحداثيات البوكسات
    box_file = DEST_BOX_LABELS / f"{file_stem}.txt"
    box_file.write_text("\n".join(yolo_lines) + "\n", encoding='utf-8')

    if count % 50 == 0 or count == len(selected_images):
        print(f"تمت معالجة {count}/{len(selected_images)} صورة...")

print("\n" + "="*50)
print("✅ اكتملت العملية بنجاح! تم تنظيم الـ 200 صورة في المجلدات التالية:")
print("1. مجلد الصور الأصلي:   ", DEST_IMAGES)
print("2. مجلد النصوص الأصلية: ", DEST_TEXT_LABELS)
print("3. مجلد إحداثيات YOLO:  ", DEST_BOX_LABELS)
print("="*50)

📁 تم إيجاد 2043 صورة. جاري معالجة ونقل 200 صورة فقط...

تمت معالجة 50/200 صورة...
تمت معالجة 100/200 صورة...
تمت معالجة 150/200 صورة...
تمت معالجة 200/200 صورة...

✅ اكتملت العملية بنجاح! تم تنظيم الـ 200 صورة في المجلدات التالية:
1. مجلد الصور الأصلي:    /content/drive/MyDrive/Data/Data_Labled/images
2. مجلد النصوص الأصلية:  /content/drive/MyDrive/Data/Data_Labled/text_labels
3. مجلد إحداثيات YOLO:   /content/drive/MyDrive/Data/Data_Labled/box_labels


In [ ]:
# خلية الضبط الدقيق (Fine-Tuning) بعد إصلاح مسارات التسميات وتسريع القراءة
import yaml
import shutil
from pathlib import Path
from ultralytics import YOLO

# 1. إنشاء مجلد كاش محلي سريع على كولاب للتدريب
LOCAL_FINETUNE_DATA = Path('/content/modi_finetune_local')
LOCAL_FT_IMAGES = LOCAL_FINETUNE_DATA / 'images'
LOCAL_FT_LABELS = LOCAL_FINETUNE_DATA / 'labels'  # اسم مجلد التسميات المطلوب لـ YOLO

for d in [LOCAL_FT_IMAGES, LOCAL_FT_LABELS]:
    d.mkdir(parents=True, exist_ok=True)

# 2. نسخ 200 صورة وتسميات البوكسات إلى التخزين المحلي
DATA_LABELED_DIR = Path('/content/drive/MyDrive/Data/Data_Labled')
DRIVE_IMAGES = DATA_LABELED_DIR / 'images'
DRIVE_BOX_LABELS = DATA_LABELED_DIR / 'box_labels'

print("🚚 جارٍ نقل البيانات إلى التخزين المحلي لتسريع التدريب وحل المشكلة...")
shutil.copytree(DRIVE_IMAGES, LOCAL_FT_IMAGES, dirs_exist_ok=True)
shutil.copytree(DRIVE_BOX_LABELS, LOCAL_FT_LABELS, dirs_exist_ok=True)

# 3. إنشاء ملف YAML بالإعدادات الصحيحة والمسار المحلي
REAL_YAML_PATH = LOCAL_FINETUNE_DATA / 'real_data.yaml'

if 'ID_TO_CLASS' in globals():
    class_names = ID_TO_CLASS
elif 'CLASS_TO_ID' in globals():
    class_names = {v: k for k, v in CLASS_TO_ID.items()}
else:
    class_names = {i: f"char_{i}" for i in range(100)}

yaml_content = {
    'path': str(LOCAL_FINETUNE_DATA),
    'train': 'images',
    'val': 'images',
    'names': class_names
}

with open(REAL_YAML_PATH, 'w', encoding='utf-8') as f:
    yaml.dump(yaml_content, f, allow_unicode=True)

# 4. تحميل النموذج وتدريبه
base_model_path = PERSISTENT_BEST if 'PERSISTENT_BEST' in globals() and PERSISTENT_BEST.exists() else BEST_CHECKPOINT

print(f"🚀 بدء Fine-Tuning باستخدام النموذج: {base_model_path}")
model = YOLO(str(base_model_path))

FINETUNE_RUNS = RUNS / 'fine_tune_real'
FINETUNE_RUNS.mkdir(parents=True, exist_ok=True)

results = model.train(
    data=str(REAL_YAML_PATH),
    epochs=30,
    imgsz=512,
    batch=8,
    lr0=0.001,
    lrf=0.01,
    freeze=10,
    device=DEVICE,
    project=str(RUNS),
    name='fine_tune_real',
    exist_ok=True
)

# 5. حفظ النسخة النهائية في Google Drive
FINETUNE_BEST_LOCAL = FINETUNE_RUNS / 'weights' / 'best.pt'
FINETUNE_BEST_DRIVE = PROJECT / 'modi_yolo11s_finetuned_real.pt'

if FINETUNE_BEST_LOCAL.exists():
    shutil.copy2(FINETUNE_BEST_LOCAL, FINETUNE_BEST_DRIVE)
    print("\n" + "="*50)
    print("🎉 اكتمل الضبط الدقيق (Fine-Tuning) بنجاح!")
    print(f"💾 تم حفظ النموذج في Google Drive:\n {FINETUNE_BEST_DRIVE}")
    print("="*50)

🚚 جارٍ نقل البيانات إلى التخزين المحلي لتسريع التدريب وحل المشكلة...
🚀 بدء Fine-Tuning باستخدام النموذج: /content/drive/MyDrive/modi_ocr_yolo11s_project/runs/modi_yolo_multiclass/weights/best.pt
Ultralytics 8.4.143 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/modi_finetune_local/real_data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=30, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=10, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=512, iou=0.7, keras=False, kobj=1

/usr/local/lib/python3.13/dist-packages/ultralytics/utils/metrics.py:607: UserWarning: Glyph 71168 (\N{MODI LETTER A}) missing from font(s) Arial Unicode MS.
  fig.savefig(plot_fname, dpi=250)
/usr/local/lib/python3.13/dist-packages/ultralytics/utils/metrics.py:607: UserWarning: Glyph 71169 (\N{MODI LETTER AA}) missing from font(s) Arial Unicode MS.
  fig.savefig(plot_fname, dpi=250)
/usr/local/lib/python3.13/dist-packages/ultralytics/utils/metrics.py:607: UserWarning: Glyph 71171 (\N{MODI LETTER II}) missing from font(s) Arial Unicode MS.
  fig.savefig(plot_fname, dpi=250)
/usr/local/lib/python3.13/dist-packages/ultralytics/utils/metrics.py:607: UserWarning: Glyph 71172 (\N{MODI LETTER U}) missing from font(s) Arial Unicode MS.
  fig.savefig(plot_fname, dpi=250)
/usr/local/lib/python3.13/dist-packages/ultralytics/utils/metrics.py:607: UserWarning: Glyph 71178 (\N{MODI LETTER E}) missing from font(s) Arial Unicode MS.
  fig.savefig(plot_fname, dpi=250)
/usr/local/lib/python3.13/dist-pa

                   all        200      12218      0.572      0.568      0.575      0.495
                     𑘀         48         84      0.344      0.857      0.627      0.532
                     𑘁         31         33      0.585      0.303       0.33      0.292
                     𑘃        171        667      0.831      0.556      0.683      0.605
                     𑘄        180       1186      0.613       0.81      0.723      0.587
                     𑘊        128        554      0.506      0.819      0.689      0.546
                     𑘋         59         91      0.717      0.495      0.556      0.504
                     𑘌         80        145      0.694      0.324      0.485      0.385
                     𑘍         48         69      0.672       0.58      0.717      0.566
                     𑘎        100        313      0.779      0.565      0.672      0.582
                     𑘏         62        104      0.769      0.481      0.566      0.521
                     

In [ ]:
# الخلية 9 — تصدير النموذج إلى ONNX وحفظ الفئات
import json
import shutil
from pathlib import Path
from ultralytics import YOLO

# 1. تحديد مسار نموذج الفين تيون المحدث
finetuned_model_path = PROJECT / 'modi_yolo11s_finetuned_real.pt'

if not finetuned_model_path.exists():
    finetuned_model_path = RUNS / 'fine_tune_real' / 'weights' / 'best.pt'

model = YOLO(str(finetuned_model_path))

# 2. تصدير النموذج إلى صيغة ONNX
onnx_file = model.export(
    format='onnx',
    imgsz=512,       # حجم الصور المعتمد بالتدريب 512
    opset=12,
    simplify=True,
    dynamic=False,
    nms=False
)
onnx_file = Path(onnx_file)

# 3. حفظ ملف ONNX وملف الفئات JSON
EXPORTS.mkdir(parents=True, exist_ok=True)
shutil.copy2(onnx_file, EXPORTS / 'modi_yolo_multiclass.onnx')

classes_json_path = EXPORTS / 'classes.json'
classes_data = model.names if hasattr(model, 'names') and model.names else {}
classes_json_path.write_text(json.dumps(classes_data, ensure_ascii=False, indent=2), encoding='utf-8')

print('Classes:', EXPORTS / 'classes.json')
print('ONNX:', EXPORTS / 'modi_yolo_multiclass.onnx')

Ultralytics 8.4.143 🚀 Python-3.13.15 torch-2.11.0+cu128 CPU (Intel Xeon CPU @ 2.00GHz)
WARNING ⚠️ This model has no one-to-one head; using one-to-many outputs.
YOLO11s summary (fused): 100 layers, 9,428,280 parameters, 0 gradients, 13.7 GFLOPs

PyTorch: starting from '/content/drive/MyDrive/modi_ocr_yolo11s_project/modi_yolo11s_finetuned_real.pt' with input shape (1, 3, 512, 512) BCHW and output shape(s) (1, 44, 5376) (18.3 MB)

ONNX: starting export with onnx 1.19.1 opset 12...
ONNX: slimming with onnxslim 0.1.96...
ONNX: export success ✅ 2.4s, saved as '/content/drive/MyDrive/modi_ocr_yolo11s_project/modi_yolo11s_finetuned_real.onnx' (36.2 MB)

Export complete (3.6s)
Results saved to /content/drive/MyDrive/modi_ocr_yolo11s_project/modi_yolo11s_finetuned_real.onnx
Predict:         yolo predict task=detect model=/content/drive/MyDrive/modi_ocr_yolo11s_project/modi_yolo11s_finetuned_real.onnx imgsz=512 
Validate:        yolo val task=detect model=/content/drive/MyDrive/modi_ocr_yolo11s_

In [ ]:
# الخلية 10 — فحص ONNX قبل بناء APK
from pathlib import Path

try:
    import onnxruntime as ort
except ImportError:
    !pip install -q onnxruntime
    import onnxruntime as ort

onnx_path = EXPORTS / 'modi_yolo_multiclass.onnx'

sess = ort.InferenceSession(str(onnx_path), providers=['CPUExecutionProvider'])

print('Inputs:', [(x.name, x.shape, x.type) for x in sess.get_inputs()])
print('Outputs:', [(x.name, x.shape, x.type) for x in sess.get_outputs()])
print("\n✅ تم فحص ONNX بنجاح وهو جاهز للتحويل لـ APK!")

Inputs: [('images', [1, 3, 512, 512], 'tensor(float)')]
Outputs: [('output0', [1, 44, 5376], 'tensor(float)')]

✅ تم فحص ONNX بنجاح وهو جاهز للتحويل لـ APK!


In [ ]:
# خلية تقييم الدقة ونسبة الخطأ على مستوى الحرف والكلمة
import json
import pandas as pd
import numpy as np

def evaluate_ocr_performance(ground_truth_list, predictions_list):
    """
    ground_truth_list: قائمة بالنصوص الحقيقية المكتوبة في الصور [gt1, gt2, ...]
    predictions_list: قائمة بالنصوص المستخرجة من النموذج [pred1, pred2, ...]
    """
    total_chars_gt = 0
    correct_chars = 0
    substitutions = 0  # استبدال حرف بحرف خاطئ

    total_words_gt = 0
    correct_words = 0
    wrong_words = 0

    word_details = []

    for gt_text, pred_text in zip(ground_truth_list, predictions_list):
        # 1. حسابات مستوى الحرف
        gt_chars = list(gt_text.replace(" ", ""))
        pred_chars = list(pred_text.replace(" ", ""))

        total_chars_gt += len(gt_chars)

        # مقارنة الحروف المتقابلة
        min_len = min(len(gt_chars), len(pred_chars))
        for c1, c2 in zip(gt_chars[:min_len], pred_chars[:min_len]):
            if c1 == c2:
                correct_chars += 1
            else:
                substitutions += 1

        # الحروف المحذوفة أو الزائدة ناتجة عن فرق الطول
        length_diff = abs(len(gt_chars) - len(pred_chars))

        # 2. حسابات مستوى الكلمة
        gt_words = gt_text.strip().split()
        pred_words = pred_text.strip().split()

        total_words_gt += len(gt_words)

        min_w_len = min(len(gt_words), len(pred_words))
        for w_gt, w_pred in zip(gt_words[:min_w_len], pred_words[:min_w_len]):
            if w_gt == w_pred:
                correct_words += 1
            else:
                wrong_words += 1
                word_details.append({"GroundTruth": w_gt, "Prediction": w_pred, "Status": "خطأ"})

        if len(pred_words) < len(gt_words):
            wrong_words += (len(gt_words) - len(pred_words))

    # الحسابات الناتجة
    wrong_chars = total_chars_gt - correct_chars
    char_accuracy = (correct_chars / total_chars_gt * 100) if total_chars_gt > 0 else 0
    cer = (wrong_chars / total_chars_gt * 100) if total_chars_gt > 0 else 0

    word_accuracy = (correct_words / total_words_gt * 100) if total_words_gt > 0 else 0
    wer = (wrong_words / total_words_gt * 100) if total_words_gt > 0 else 0

    print("=" * 50)
    print("📊 تقرير تقييم دقة النموذج (OCR Evaluation Report)")
    print("=" * 50)
    print(f"🔤 **على مستوى الحرف (Character-Level):**")
    print(f"  - إجمالي الحروف الحقيقية: {total_chars_gt}")
    print(f"  - التوقعات الصحيحة: {correct_chars} ({char_accuracy:.2f}%)")
    print(f"  - التوقعات الخاطئة: {wrong_chars} (معدل الخطأ CER: {cer:.2f}%)")
    print("-" * 50)
    print(f"📝 **على مستوى الكلمة (Word-Level):**")
    print(f"  - إجمالي الكلمات الحقيقية: {total_words_gt}")
    print(f"  - الكلمات الصحيحة بالكامل: {correct_words} ({word_accuracy:.2f}%)")
    print(f"  - الكلمات الخاطئة: {wrong_words} (معدل الخطأ WER: {wer:.2f}%)")
    print("=" * 50)

    return {
        "char_accuracy": char_accuracy,
        "cer": cer,
        "word_accuracy": word_accuracy,
        "wer": wer,
        "correct_chars": correct_chars,
        "wrong_chars": wrong_chars,
        "correct_words": correct_words,
        "wrong_words": wrong_words
    }

# مثال لاستخدام الدالة مع بيانات تجريبية:
ground_truths = ["𑘦𑘻𑘟 me", "𑘭𑘝𑘿𑘧"]
predictions   = ["𑘦𑘻𑘟 e",  "𑘭𑘝𑘿𑘧"]
evaluate_ocr_performance(ground_truths, predictions)

📊 تقرير تقييم دقة النموذج (OCR Evaluation Report)
🔤 **على مستوى الحرف (Character-Level):**
  - إجمالي الحروف الحقيقية: 9
  - التوقعات الصحيحة: 7 (77.78%)
  - التوقعات الخاطئة: 2 (معدل الخطأ CER: 22.22%)
--------------------------------------------------
📝 **على مستوى الكلمة (Word-Level):**
  - إجمالي الكلمات الحقيقية: 3
  - الكلمات الصحيحة بالكامل: 2 (66.67%)
  - الكلمات الخاطئة: 1 (معدل الخطأ WER: 33.33%)


{'char_accuracy': 77.77777777777779,
 'cer': 22.22222222222222,
 'word_accuracy': 66.66666666666666,
 'wer': 33.33333333333333,
 'correct_chars': 7,
 'wrong_chars': 2,
 'correct_words': 2,
 'wrong_words': 1}

In [ ]:
import onnxruntime as ort

model_path = str(EXPORTS / "modi_yolo_multiclass.onnx")

sess = ort.InferenceSession(
    model_path,
    providers=["CPUExecutionProvider"]
)

print("INPUTS:")
for x in sess.get_inputs():
    print("name =", x.name)
    print("shape =", x.shape)
    print("type =", x.type)

print("\nOUTPUTS:")
for x in sess.get_outputs():
    print("name =", x.name)
    print("shape =", x.shape)
    print("type =", x.type)


INPUTS:
name = images
shape = [1, 3, 512, 512]
type = tensor(float)

OUTPUTS:
name = output0
shape = [1, 44, 5376]
type = tensor(float)
